In [ ]:
# Instalamos gymnasium, la librería que contiene el entorno FrozenLake
!pip install gymnasium[toy_text] --quiet

In [ ]:
import gymnasium as gym      # contiene el entorno FrozenLake y su lógica
import numpy as np           # lo usamos para crear y operar la Q-Table
import matplotlib.pyplot as plt  # lo usamos para graficar los resultados
import matplotlib.patches as mpatches  # para dibujar los cuadros del tablero
import random                # para generar números aleatorios en la exploración
import warnings
warnings.filterwarnings('ignore')  # ocultamos advertencias de versiones

# Fijamos semillas para que los resultados sean iguales cada vez que se corra
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Creamos el entorno FrozenLake
# is_slippery=False: el agente se mueve exactamente a donde decide (sin resbalar)
# render_mode=None: no mostramos animación visual durante el entrenamiento (más rápido)
env = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)

# Número de estados posibles: 16, porque el tablero es 4x4 y cada celda es un estado
n_states = env.observation_space.n

# Número de acciones posibles: 4 (izquierda, abajo, derecha, arriba)
n_actions = env.action_space.n

# Diccionario para traducir el número de acción a texto legible
acciones = {0: '← Izquierda', 1: '↓ Abajo', 2: '→ Derecha', 3: '↑ Arriba'}

print(f'Estados: {n_states} | Acciones: {n_actions}')

In [ ]:
# Dibujamos el tablero del entorno para entender el problema
# S = Start (inicio), F = Frozen (hielo seguro), H = Hole (hoyo = pierde), G = Goal (meta)
mapa = [
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G']
]

# Asignamos un color a cada tipo de celda
colores = {'S': '#3498db', 'F': '#d6eaf8', 'H': '#2c3e50', 'G': '#2ecc71'}

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_title('Entorno FrozenLake 4×4', fontsize=13, fontweight='bold', pad=12)

for fila in range(4):
    for col in range(4):
        celda = mapa[fila][col]              # tipo de celda en esa posición
        estado = fila * 4 + col              # número de estado (0 a 15)
        color = colores[celda]               # color según el tipo

        # Dibujamos el rectángulo de la celda
        rect = mpatches.FancyBboxPatch(
            (col + 0.05, 3 - fila + 0.05), 0.9, 0.9,
            boxstyle='round,pad=0.05', color=color
        )
        ax.add_patch(rect)

        # Escribimos el símbolo de la celda (S, F, H, G)
        ax.text(col + 0.5, 3 - fila + 0.6, celda,
                ha='center', va='center', fontsize=18, fontweight='bold',
                color='white' if celda == 'H' else '#2c3e50')

        # Escribimos el número de estado en la esquina inferior de la celda
        ax.text(col + 0.85, 3 - fila + 0.15, str(estado),
                ha='center', va='center', fontsize=8, color='gray')

ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.axis('off')

# Leyenda explicativa
leyenda = [
    mpatches.Patch(color='#3498db', label='S: Inicio'),
    mpatches.Patch(color='#d6eaf8', label='F: Hielo seguro'),
    mpatches.Patch(color='#2c3e50', label='H: Hoyo (pierde)'),
    mpatches.Patch(color='#2ecc71', label='G: Meta (gana)'),
]
ax.legend(handles=leyenda, loc='upper right', bbox_to_anchor=(1.65, 1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Hiperparámetros: valores que nosotros definimos antes de entrenar

N_EPISODIOS   = 5000   # número de partidas que jugará el agente para aprender
MAX_PASOS     = 100    # máximo de pasos por partida para evitar bucles infinitos
ALPHA         = 0.8    # tasa de aprendizaje: qué tanto peso tiene la información nueva (0 a 1)
GAMMA         = 0.95   # factor de descuento: qué tanto valora el agente las recompensas futuras (0 a 1)
EPSILON_INI   = 1.0    # epsilon inicial: 100% exploración al comienzo (acciones totalmente aleatorias)
EPSILON_MIN   = 0.01   # epsilon mínimo: siempre se explora al menos un 1%
EPSILON_DECAY = 0.001  # cuánto decrece epsilon en cada episodio (el agente explora menos con el tiempo)

In [ ]:
# Creamos la Q-Table: una matriz de (estados x acciones) con todos los valores en 0
# Cada celda Q[s, a] representará cuánta recompensa futura espera el agente
# si estando en el estado s toma la acción a
Q_table = np.zeros((n_states, n_actions))

print('Q-Table inicial (el agente no sabe nada todavía):')
print(Q_table)

In [ ]:
# Listas para guardar estadísticas del entrenamiento
recompensas_por_episodio = []  # recompensa obtenida en cada episodio (1 = éxito, 0 = falla)
epsilon_por_episodio = []      # valor de epsilon en cada episodio (para graficar su decaimiento)

epsilon = EPSILON_INI  # epsilon comienza en 1.0 (exploración total)

for episodio in range(N_EPISODIOS):

    # Reiniciamos el entorno: el agente vuelve al estado 0 (celda S)
    estado, _ = env.reset(seed=SEED)

    recompensa_total = 0  # acumulador de recompensa en este episodio
    terminado = False     # indica si el episodio terminó (llegó a G o cayó en H)

    for paso in range(MAX_PASOS):

        # Estrategia epsilon-greedy: decidimos si explorar o explotar
        if random.uniform(0, 1) < epsilon:
            # EXPLORACIÓN: elegimos una acción al azar para descubrir cosas nuevas
            accion = env.action_space.sample()
        else:
            # EXPLOTACIÓN: elegimos la acción con el mayor valor Q en el estado actual
            accion = np.argmax(Q_table[estado, :])

        # Ejecutamos la acción en el entorno y recibimos el resultado
        # nuevo_estado: celda a la que llegó el agente
        # recompensa: 1 si llegó a G, 0 en cualquier otro caso
        # terminado: True si llegó a G o cayó en H
        nuevo_estado, recompensa, terminado, truncado, _ = env.step(accion)

        # Actualizamos la Q-Table con la ecuación de Bellman:
        # Q(s,a) ← Q(s,a) + α * [r + γ * max(Q(s',a')) - Q(s,a)]
        mejor_q_futuro = np.max(Q_table[nuevo_estado, :])  # el mejor valor Q desde el nuevo estado
        Q_table[estado, accion] = Q_table[estado, accion] + ALPHA * (
            recompensa + GAMMA * mejor_q_futuro - Q_table[estado, accion]
        )

        # Avanzamos: el nuevo estado pasa a ser el estado actual
        estado = nuevo_estado
        recompensa_total += recompensa

        # Si el episodio terminó (cayó en H o llegó a G), salimos del bucle de pasos
        if terminado or truncado:
            break

    # Reducimos epsilon: el agente irá explorando menos y explotando más con el tiempo
    epsilon = max(EPSILON_MIN, epsilon - EPSILON_DECAY)

    # Guardamos estadísticas del episodio
    recompensas_por_episodio.append(recompensa_total)
    epsilon_por_episodio.append(epsilon)

    # Mostramos el progreso cada 1000 episodios
    if (episodio + 1) % 1000 == 0:
        promedio = np.mean(recompensas_por_episodio[-1000:]) * 100
        print(f'Episodio {episodio+1:5d}/{N_EPISODIOS} | Éxito últimos 1000: {promedio:.1f}% | ε={epsilon:.4f}')

env.close()
print('\n¡Entrenamiento completado!')

In [ ]:
# Graficamos la tasa de éxito y el decaimiento de epsilon durante el entrenamiento

# Calculamos el promedio móvil de 100 episodios para suavizar la curva
ventana = 100
exito_promedio = [
    np.mean(recompensas_por_episodio[max(0, i - ventana):i + 1])
    for i in range(len(recompensas_por_episodio))
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Progreso del entrenamiento', fontsize=13, fontweight='bold')

# Gráfica izquierda: tasa de éxito por episodio
axes[0].plot(exito_promedio, color='#2ecc71', linewidth=1.5)
axes[0].fill_between(range(len(exito_promedio)), exito_promedio, alpha=0.2, color='#2ecc71')
axes[0].set_xlabel('Episodio')
axes[0].set_ylabel('Tasa de éxito (1 = llega a la meta, 0 = cae en hoyo)')
axes[0].set_title('Tasa de éxito (promedio móvil de 100 episodios)')
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Gráfica derecha: decaimiento de epsilon
axes[1].plot(epsilon_por_episodio, color='#e74c3c', linewidth=1.5)
axes[1].fill_between(range(len(epsilon_por_episodio)), epsilon_por_episodio, alpha=0.2, color='#e74c3c')
axes[1].set_xlabel('Episodio')
axes[1].set_ylabel('Valor de epsilon')
axes[1].set_title('Decaimiento de epsilon (exploración → explotación)')
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Mostramos la Q-Table aprendida como heatmap
# Los colores más intensos indican que el agente prefiere esa acción en ese estado

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(Q_table, cmap='YlOrRd', aspect='auto')

ax.set_title('Q-Table aprendida después del entrenamiento', fontsize=12, fontweight='bold')
ax.set_xlabel('Acción')
ax.set_ylabel('Estado')
ax.set_xticks(range(n_actions))
ax.set_xticklabels(['← Izq', '↓ Abajo', '→ Der', '↑ Arriba'])
ax.set_yticks(range(n_states))
ax.set_yticklabels([f's={i}' for i in range(n_states)], fontsize=8)

# Escribimos el valor numérico dentro de cada celda del heatmap
for i in range(n_states):
    for j in range(n_actions):
        valor = Q_table[i, j]
        color_texto = 'white' if valor > Q_table.max() * 0.6 else 'black'
        ax.text(j, i, f'{valor:.3f}', ha='center', va='center', fontsize=8, color=color_texto)

plt.colorbar(im, ax=ax, label='Valor Q (mayor = el agente prefiere esa acción en ese estado)')
plt.tight_layout()
plt.show()

In [ ]:
# Dibujamos la política aprendida: qué acción tomará el agente en cada celda del tablero

mapa = [
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G']
]
colores = {'S': '#3498db', 'F': '#d6eaf8', 'H': '#2c3e50', 'G': '#2ecc71'}
flechas = {0: '←', 1: '↓', 2: '→', 3: '↑'}  # traducción de número de acción a flecha

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.set_title('Política aprendida (acción óptima por estado)', fontsize=12, fontweight='bold', pad=12)

for fila in range(4):
    for col in range(4):
        celda = mapa[fila][col]
        estado = fila * 4 + col
        color = colores[celda]

        # Dibujamos el cuadro de la celda
        rect = mpatches.FancyBboxPatch(
            (col + 0.05, 3 - fila + 0.05), 0.9, 0.9,
            boxstyle='round,pad=0.05', color=color, zorder=1
        )
        ax.add_patch(rect)

        # Escribimos el símbolo de la celda
        ax.text(col + 0.5, 3 - fila + 0.75, celda,
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if celda == 'H' else '#2c3e50', zorder=2)

        # En las celdas que no son hoyo ni meta, mostramos la acción óptima como flecha
        if celda not in ('H', 'G'):
            accion_optima = np.argmax(Q_table[estado, :])  # acción con mayor valor Q
            ax.text(col + 0.5, 3 - fila + 0.35, flechas[accion_optima],
                    ha='center', va='center', fontsize=22, color='#2c3e50',
                    fontweight='bold', zorder=2)

        # Número de estado en la esquina
        ax.text(col + 0.15, 3 - fila + 0.15, str(estado),
                ha='center', va='center', fontsize=8, color='gray', zorder=2)

ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.axis('off')

leyenda = [
    mpatches.Patch(color='#3498db', label='S: Inicio'),
    mpatches.Patch(color='#d6eaf8', label='F: Hielo (flecha = acción óptima)'),
    mpatches.Patch(color='#2c3e50', label='H: Hoyo'),
    mpatches.Patch(color='#2ecc71', label='G: Meta'),
]
ax.legend(handles=leyenda, loc='lower center', bbox_to_anchor=(0.5, -0.12), fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Corremos una demostración: el agente entrenado juega una partida completa
# Ya no usa exploración (epsilon=0): solo elige la mejor acción de la Q-Table

env_demo = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)
estado, _ = env_demo.reset(seed=SEED)
terminado = False
pasos_demo = []  # guardamos cada paso para visualizar el recorrido

print('Demostración del agente entrenado:')
print('=' * 50)

paso_num = 0
while not terminado and paso_num < MAX_PASOS:
    # El agente elige la acción con mayor valor Q (sin aleatoriedad)
    accion = np.argmax(Q_table[estado, :])

    fila_actual, col_actual = estado // 4, estado % 4  # convertimos estado a posición (fila, col)

    nuevo_estado, recompensa, terminado, truncado, _ = env_demo.step(accion)

    nueva_fila, nueva_col = nuevo_estado // 4, nuevo_estado % 4

    print(f'  Paso {paso_num+1}: estado {estado:2d} (fila={fila_actual}, col={col_actual}) '
          f'→ acción: {acciones[accion]:13s} '
          f'→ nuevo estado {nuevo_estado:2d} (fila={nueva_fila}, col={nueva_col})')

    pasos_demo.append({'estado': estado, 'accion': accion,
                       'nuevo_estado': nuevo_estado, 'recompensa': recompensa})
    estado = nuevo_estado
    paso_num += 1

print('=' * 50)
if recompensa == 1:
    print(f'\n¡El agente llegó a la meta en {paso_num} pasos!')
else:
    print(f'\nEl agente cayó en un hoyo.')

env_demo.close()

In [ ]:
# Dibujamos visualmente el camino que recorrió el agente en la demostración

mapa = [
    ['S', 'F', 'F', 'F'],
    ['F', 'H', 'F', 'H'],
    ['F', 'F', 'F', 'H'],
    ['H', 'F', 'F', 'G']
]
colores_base = {'S': '#3498db', 'F': '#d6eaf8', 'H': '#2c3e50', 'G': '#2ecc71'}

# Lista de todos los estados visitados (incluyendo el estado final)
estados_visitados = [p['estado'] for p in pasos_demo] + [pasos_demo[-1]['nuevo_estado']]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.set_title(f'Recorrido del agente ({paso_num} pasos)', fontsize=12, fontweight='bold', pad=12)

for fila in range(4):
    for col in range(4):
        celda = mapa[fila][col]
        estado = fila * 4 + col
        color = colores_base[celda]

        # Resaltamos en morado las celdas que el agente visitó
        if estado in estados_visitados and celda != 'H':
            if estado == estados_visitados[0]:
                color = '#3498db'   # azul para el inicio
            elif estado == estados_visitados[-1] and recompensa == 1:
                color = '#f39c12'   # naranja para la meta alcanzada
            else:
                color = '#9b59b6'   # morado para el camino recorrido

        rect = mpatches.FancyBboxPatch(
            (col + 0.05, 3 - fila + 0.05), 0.9, 0.9,
            boxstyle='round,pad=0.05', color=color, zorder=1
        )
        ax.add_patch(rect)

        # Símbolo de la celda
        ax.text(col + 0.5, 3 - fila + 0.65, celda,
                ha='center', va='center', fontsize=12, fontweight='bold',
                color='white' if celda == 'H' else '#2c3e50', zorder=2)

        # Número del paso en que se visitó la celda
        if estado in estados_visitados:
            orden = [i for i, e in enumerate(estados_visitados) if e == estado]
            if orden:
                ax.text(col + 0.5, 3 - fila + 0.28, f'paso {orden[0]}',
                        ha='center', va='center', fontsize=8,
                        color='white' if celda == 'H' else '#7f8c8d', zorder=2)

ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.axis('off')

leyenda = [
    mpatches.Patch(color='#3498db', label='Inicio'),
    mpatches.Patch(color='#9b59b6', label='Camino recorrido'),
    mpatches.Patch(color='#f39c12', label='Meta alcanzada'),
    mpatches.Patch(color='#2c3e50', label='Hoyo (evitado)'),
]
ax.legend(handles=leyenda, loc='lower center', bbox_to_anchor=(0.5, -0.12), fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación final: hacemos jugar al agente 1000 partidas sin ninguna exploración
# para medir qué tan bien aprendió

env_eval = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)
n_pruebas = 1000
exitos = 0

for _ in range(n_pruebas):
    estado, _ = env_eval.reset()
    terminado = False
    for _ in range(MAX_PASOS):
        # Solo explotación: siempre la mejor acción según la Q-Table
        accion = np.argmax(Q_table[estado, :])
        estado, recompensa, terminado, truncado, _ = env_eval.step(accion)
        if terminado or truncado:
            break
    if recompensa == 1:  # si la última recompensa fue 1, llegó a la meta
        exitos += 1

env_eval.close()

tasa = (exitos / n_pruebas) * 100
print(f'Partidas jugadas : {n_pruebas}')
print(f'Partidas ganadas : {exitos}')
print(f'Tasa de éxito    : {tasa:.1f}%')